# Modelado de Fatiga con Time Series Transformer

Este notebook contiene la explicación teórica, la revisión de literatura científica, la arquitectura detallada y la implementación paso a paso de un **Time Series Transformer** en PyTorch (empleando codificación posicional sinusoidal y bloques de auto-atención multi-cabeza construidos de forma manual) para predecir los niveles continuos de fatiga física y mental del dataset **FatigueSet**.

---

## 1. Fundamentos Teóricos y Literatura de Referencia

Las redes recurrentes (LSTM/GRU) y convolucionales (TCN) procesan las secuencias explotando sesgos inductivos de localidad temporal o recurrencia paso a paso. El **Transformer** (Vaswani et al., 2017) propone un paradigma basado exclusivamente en mecanismos de **atención** global, eliminando la necesidad de recurrencia convolucional o secuencial y permitiendo una paralelización completa.

### Codificación Posicional Sinusoidal

Dado que los mecanismos de atención son invariantes frente al orden secuencial, debemos inyectar información sobre la posición temporal de cada elemento de la serie. Para ello, sumamos un vector de codificación posicional estático a la proyección lineal del input:
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
Donde $pos$ es la posición de la secuencia (timestep) e $i$ es el índice del canal de embedding.

### Auto-Atención Multi-Cabeza (Multi-Head Self-Attention)

El mecanismo de atención permite al modelo ponderar dinámicamente la relevancia de diferentes timesteps para construir una representación contextualizada. La atención individual de cada cabeza se calcula mediante el producto punto escalado:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$
Donde:
- $Q = X W_q, \quad K = X W_k, \quad V = X W_v$ son las proyecciones Query, Key y Value del tensor de entrada $X$.
- $d_k = d_{model} / H$ es la dimensión de cada cabeza de atención ($H$ es el número de cabezas).

La auto-atención multi-cabeza concatena los resultados de las cabezas y aplica una proyección final de salida:
$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_H) W_o$$

### Estructura del Codificador y Agregación

Cada bloque del codificador consta de dos subcapas principales: auto-atención multi-cabeza y una red feed-forward de dos niveles (FFN). Cada una de estas subcapas emplea una conexión residual seguida de normalización de capa (Post-LayerNorm):
$$x_1 = \text{LayerNorm}(x + \text{Dropout}(\text{SelfAttn}(x)))$$
$$x_2 = \text{LayerNorm}(x_1 + \text{Dropout}(\text{FFN}(x_1)))$$

La salida final del codificador $Y \in \mathbb{R}^{B \times L \times d_{model}}$ se agrega mediante **Global Average Pooling** a lo largo de la dimensión temporal:
$$y_{mean} = \frac{1}{L} \sum_{t=1}^{L} Y_{:, t, :}$$
Y se proyecta con una capa lineal final $\mathbb{R}^{d_{model}} \rightarrow \mathbb{R}^2$ para estimar la fatiga.

---

### Diagrama de Flujo del Transformer Encoder (Mermaid)

```mermaid
graph TD
    subgraph Entrada
        in["Input Secuencia: (B, seq_len, input_size)"]
    end
    
    subgraph "Proyección e Inyección Temporal"
        proj["Proyección Lineal: (B, seq_len, d_model)"]
        pe["Codificación Posicional Sinusoidal (PE)"]
        add_pe["Suma: Proyección + PE"]
        in --> proj
        proj --> add_pe
        pe --> add_pe
    end

    subgraph "Codificador Custom (Bloque Encoder)"
        mha["CustomMultiHeadAttention (H cabezas)"]
        add_norm1["Add & LayerNorm 1"]
        ffn["Feed Forward Network (Linear -> ReLU -> Linear)"]
        add_norm2["Add & LayerNorm 2"]
        
        add_pe --> mha
        mha --> add_norm1
        add_pe -.->|Conexión Residual| add_norm1
        add_norm1 --> ffn
        ffn --> add_norm2
        add_norm1 -.->|Conexión Residual| add_norm2
    end

    subgraph "Predicción Regresora"
        pool["Global Average Pooling (dim=1): (B, d_model)"]
        fc["Capa Lineal de Regresión: (B, 2)"]
        
        add_norm2 --> pool
        pool --> fc
    end
```

---

### Citas Bibliográficas Científicas

* **Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., ... & Polosukhin, I. (2017).** *Attention Is All You Need*. Advances in Neural Information Processing Systems (NeurIPS), 30, 5998-6008. [Enlace al Paper](https://arxiv.org/abs/1706.03762)
* **Lim, B., & Zohren, S. (2021).** *Time-series forecasting with deep learning: a survey*. Philosophical Transactions of the Royal Society A, 379(2194), 20200209. [Enlace al Paper](https://royalsocietypublishing.org/doi/10.1098/rsta.2020.0209)

In [1]:
# SETUP e IMPORTACIONES
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Añadir fatigueset-lib al sys.path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset import FatigueSetPipeline
from fatigueset.models import CustomTSTransformerRegressor, FatigueSequenceDataset
from fatigueset.models.rnn import _prepare_target_table, _merge_raw_streams, _build_sequences

print("[OK] Imports completados y path configurado.")
print(f"Dispositivo actual: {'cuda' if torch.cuda.is_available() else 'cpu'}")

[OK] Imports completados y path configurado.
Dispositivo actual: cuda


## 2. Configuración del Pipeline y Construcción de Secuencias

Cargamos los datos fisiológicos del dataset `fatigueset`, combinando las señales crudas para generar ventanas temporales de 128 instantes de tiempo.

In [2]:
# Configuración del dataset y pipeline
dataset_path = str(Path.cwd().parent / "fatigueset")
pipeline = FatigueSetPipeline(dataset_path=dataset_path, umbral_nulos=5.0)

print("Cargando dataset...")
raw = pipeline.cargar_dataset(verbose=False)

print("Preparando targets del dataframe ML...")
df_ml = pipeline.construir_dataset_ml(raw)
df_targets = _prepare_target_table(df_ml)

print("Combinando streams fisiológicos crudos (Chest y Wrist)...")
df_raw = _merge_raw_streams(raw)

# Parámetros de ventanas de secuencia temporal
seq_len = 128
step = 32

print(f"Construyendo secuencias de tamaño={seq_len} y paso={step}...")
X_arr, y_arr, groups, feature_cols = _build_sequences(
    df_raw=df_raw,
    df_targets=df_targets,
    seq_len=seq_len,
    step=step
)

print(f"[OK] Dimensiones de tensores construidos:")
print(f"  - X: {X_arr.shape} (Número de ventanas x seq_len x features)")
print(f"  - y: {y_arr.shape} (Número de ventanas x 2 targets)")
print(f"  - Columnas de sensores: {len(feature_cols)}")

Cargando dataset...
Preparando targets del dataframe ML...
Combinando streams fisiológicos crudos (Chest y Wrist)...
Construyendo secuencias de tamaño=128 y paso=32...
[OK] Dimensiones de tensores construidos:
  - X: (1306, 128, 23) (Número de ventanas x seq_len x features)
  - y: (1306, 2) (Número de ventanas x 2 targets)
  - Columnas de sensores: 23


## 3. División de Datos por Participante (Group Split)

Dividimos los datos de manera que el participante `'01'` se mantenga en el subconjunto de validación para evitar data leakage.

In [3]:
train_idx = np.where(groups != '01')[0]
val_idx = np.where(groups == '01')[0]

X_train, y_train = X_arr[train_idx], y_arr[train_idx]
X_val, y_val = X_arr[val_idx], y_arr[val_idx]

train_dataset = FatigueSequenceDataset(X_train, y_train)
val_dataset = FatigueSequenceDataset(X_val, y_val)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Train samples: 1184
Validation samples: 122


## 4. Inicialización del Regresor Transformer

Instanciamos nuestro regresor de Transformer. Configuramos $d_{model} = 64$, 4 cabezas de atención, y 2 bloques codificadores.

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = len(feature_cols)
d_model = 64
num_heads = 4
num_layers = 2
dim_feedforward = 128
dropout = 0.1

model = CustomTSTransformerRegressor(
    input_size=input_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    dim_feedforward=dim_feedforward,
    dropout=dropout,
    output_size=2
).to(device)

print(model)

CustomTSTransformerRegressor(
  (input_projection): Linear(in_features=23, out_features=64, bias=True)
  (pos_encoder): PositionalEncoding()
  (encoder_layers): ModuleList(
    (0-1): 2 x CustomTransformerEncoderLayer(
      (self_attn): CustomMultiHeadAttention(
        (q_linear): Linear(in_features=64, out_features=64, bias=True)
        (k_linear): Linear(in_features=64, out_features=64, bias=True)
        (v_linear): Linear(in_features=64, out_features=64, bias=True)
        (out_linear): Linear(in_features=64, out_features=64, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (linear1): Linear(in_features=64, out_features=128, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=128, out_features=64, bias=True)
      (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropou

## 5. Entrenamiento de Validación (5 Épocas)

Entrenamos el modelo durante 5 épocas empleando un optimizador Adam, una tasa de aprendizaje de $10^{-3}$ y gradient clipping para mantener la estabilidad del entrenamiento.

In [5]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
print("Iniciando entrenamiento...")

for epoch in range(1, epochs + 1):
    # Modo entrenamiento
    model.train()
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Modo validación
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
    
    print(f"Epoch {epoch}/{epochs} - Train Loss (MSE): {avg_train_loss:.6f} - Val Loss (MSE): {avg_val_loss:.6f}")

print("[OK] Entrenamiento finalizado correctamente.")

Iniciando entrenamiento...
Epoch 1/5 - Train Loss (MSE): 1142.386938 - Val Loss (MSE): 1055.662735
Epoch 2/5 - Train Loss (MSE): 994.295341 - Val Loss (MSE): 871.961136
Epoch 3/5 - Train Loss (MSE): 843.107458 - Val Loss (MSE): 677.230598
Epoch 4/5 - Train Loss (MSE): 689.768990 - Val Loss (MSE): 490.651749
Epoch 5/5 - Train Loss (MSE): 561.670236 - Val Loss (MSE): 328.787575
[OK] Entrenamiento finalizado correctamente.


## 6. Serialización del Modelo

Guardamos los pesos del modelo en el directorio `/models/deep_learning/`.

In [6]:
output_dir = Path.cwd().parent / "models" / "deep_learning"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "transformer_fatigue_notebook.pt"
torch.save(model.state_dict(), model_path)

print(f"[OK] Modelo guardado exitosamente en: {model_path}")

[OK] Modelo guardado exitosamente en: c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\models\deep_learning\transformer_fatigue_notebook.pt
